In [1]:
import sys , os
import pandas as pd
import re

sys.path.append('..')

from utils.prompts import render
from utils.llm_client import LLMClient
from utils.logging_utils import log_llm_call
from utils.router import pick_model, should_use_reasoning_model
from IPython.display import Markdown, display

# Part 01

In [1]:
file_path = '../data/Sample Messages.txt'
with open(file_path, 'r', encoding='utf-8') as f:
    raw_text = f.read()


cleaned_text = re.sub(r'\\s*', '', raw_text)
messages_list = [line.strip() for line in cleaned_text.split('\n') if line.strip()]

print(f"Total messages to process: {len(messages_list)}\n")


model = pick_model('google', 'general')
client = LLMClient('google', model)


results = []

examples = """
Example 1:
Message: "Breaking News: Kelani River level at 9m."
Output: District: Colombo | Intent: Info | Priority: Low

Example 2:
Message: "We are trapped on the roof with 3 kids!"
Output: District: None | Intent: Rescue | Priority: High

Example 3:
Message: "Does anyone have extra dry rations for the camp in Gampaha?"
Output: District: Gampaha | Intent: Supply | Priority: High

Example 4:
Message: "Just arrived in Galle. Weather is nice."
Output: District: Galle | Intent: Other | Priority: Low
"""


for msg in messages_list[:4]:
    

    prompt_text, spec = render(
        'few_shot.v1',
        role='rescue, supply, info, other classifier',
        examples=examples,
        query=f'Message: "{msg}"',
        constraints='Follow the pattern in examples: provide output in the format "District: <district> | Intent: <intent> | Priority: <priority>". If the district is not mentioned, use "None".',
        format='District: {{district}} | Intent: {{intent}} | Priority: {{priority}}'
    )

    chat_messages = [{'role': 'user', 'content': prompt_text}]
    

    response = client.chat(chat_messages, temperature=0.1) 
    

    output_text = response['text'].strip()
    

    results.append({
        'Original Message': msg,
        'Classification': output_text
    })
    
    print(f"Processed: {msg[:40]}... -> {output_text}")
    

    if 'latency_ms' in response:
        log_llm_call('gemini', model, 'few_shot', response['latency_ms'], response.get('usage', {}))


os.makedirs('../output', exist_ok=True)
df = pd.DataFrame(results)


output_file = '../output/classified_messages.xlsx'
df.to_excel(output_file, index=False)

print(f"\nDone! All results saved to {output_file}")

NameError: name 're' is not defined

# Part 02

In [2]:
import re
from IPython.display import Markdown, display

print("--- Starting Part 2: Stability Experiment ---\n")


file_path = '../data/Scenarios.txt'
with open(file_path, 'r', encoding='utf-8') as f:
    raw_scenarios = f.read()


cleaned_scenarios = re.sub(r'\\s*', '', raw_scenarios)


scenarios_list = [s.strip() for s in cleaned_scenarios.split('SCENARIO') if s.strip()]
scenarios_list = ['SCENARIO ' + s for s in scenarios_list]

print(f"Found {len(scenarios_list)} scenarios to test.\n")


reasoning_model = pick_model('google', 'cot') 
print(f'Using reasoning model: {reasoning_model}\n')
client_reasoning = LLMClient('google', reasoning_model)


for scenario in scenarios_list:
    print("="*80)
    print(f"Testing Scenario:\n{scenario[:150]}...\n") 
    print("="*80)
    

    prompt_text, spec = render(
        'cot_reasoning.v1', 
        role='expert emergency response coordinator',
        problem=scenario
    )
    
    chat_messages = [{'role': 'user', 'content': prompt_text}]
    
    # ---------------------------------------------------------
    # SAFE MODE (Temperature 0.0) 
    # ---------------------------------------------------------
    print("\n🟢 [ SAFE MODE | Temperature = 0.0 ]\n")
    response_safe = client_reasoning.chat(
        chat_messages, 
        temperature=0.0, 
        max_tokens=spec.max_tokens
    )
    display(Markdown(response_safe['text']))
    
    # ---------------------------------------------------------
    # CHAOS MODE (Temperature 1.0) - Running 3 times
    # ---------------------------------------------------------
    print("\n\n🔴 [ CHAOS MODE | Temperature = 1.0 ] (Running 3 times)")
    print("-" * 60)
    
    for i in range(1, 4):
        print(f"\n--- Chaos Run {i} ---")
        response_chaos = client_reasoning.chat(
            chat_messages, 
            temperature=1.0, 
            max_tokens=spec.max_tokens
        )
        display(Markdown(response_chaos['text']))
        
    print("\n" + "#"*80 + "\n")

--- Starting Part 2: Stability Experiment ---

Found 2 scenarios to test.

Using reasoning model: gemini-2.5-flash

Testing Scenario:
SCENARIO A: THE KANDY LANDSLIDE
Location: Hanthana Tea Factory
Details: "We are trapped. The access road is blocked. My uncle is stuck in the line roo...


🟢 [ SAFE MODE | Temperature = 0.0 ]



**Reasoning Steps:**

1.  **Prioritize Immediate Life & Health Threats:** Identify the uncle (imminent drowning risk) and the diabetic patient (critical medical need) as top priorities.
2.  **Simultaneous Resource Deployment:** Initiate parallel responses for rescue, medical aid, and logistical assessment.
3.  **Access & Logistics Assessment:** Determine the quickest way to reach both locations, considering the blocked road.
4.  **Communication & Coordination:** Establish clear communication channels with on-site personnel and all responding agencies.
5.  **Sustained Support Planning:** Address secondary needs (hunger, shelter) once immediate threats are managed.

---

**Answer:**

**Immediate Actions (Within 0-30 minutes):**

1.  **Emergency Activation & Prioritization:**
    *   **Uncle (Immediate Life Threat):** Immediately dispatch a specialized swift-water/landslide rescue team with appropriate equipment (boats, ropes, medical personnel) to the line rooms below the factory. Emphasize urgency due to rising water.
    *   **Diabetic Patient (Immediate Health Threat):** Simultaneously dispatch a medical team with insulin, glucose monitoring equipment, and emergency medical supplies directly to the Hanthana Tea Factory.

2.  **Access & Logistics Assessment:**
    *   **Road Blockage:** Immediately deploy an assessment team (e.g., engineering, local authorities) to evaluate the blocked access road. Determine the type of blockage, estimated clearance time, and feasibility of alternative access routes (e.g., foot teams, drone delivery for initial medical supplies, or helicopter if terrain permits and assets are available).
    *   **Communication:** Establish direct communication with the trapped individuals inside the factory to gather more precise details on the diabetic patient's condition and the uncle's exact location/status.

3.  **Coordination & Command:**
    *   Establish an Incident Command Post (ICP) at the nearest accessible safe location.
    *   Coordinate with local police, fire, medical services, and disaster management authorities.

**Subsequent Actions (Once Immediate Threats are Addressed):**

1.  **Sustained Support for Trapped Individuals:**
    *   Once medical personnel reach the factory, ensure the diabetic patient is stabilized and monitored.
    *   Coordinate delivery of food, water, and basic necessities for the 40 hungry individuals via the cleared road or alternative access.
    *   Provide psychological first aid and ongoing support to all affected individuals.
2.  **Infrastructure & Safety Assessment:**
    *   Conduct a full structural assessment of the factory and surrounding areas for further landslide risks or damage.
    *   Plan for safe evacuation or continued sheltering as appropriate.



🔴 [ CHAOS MODE | Temperature = 1.0 ] (Running 3 times)
------------------------------------------------------------

--- Chaos Run 1 ---


**Reasoning Steps:**

1.  **Prioritize Immediate Life Threats:** The uncle climbing a tree due to rising water is an imminent threat to life and takes absolute first priority.
2.  **Address Immediate Health Threats:** The collapsed diabetic patient needing insulin is a critical medical emergency requiring immediate attention.
3.  **Assess and Address Access:** The blocked access road impacts all rescue and supply efforts and needs to be assessed for potential partial access (foot/air) and longer-term clearing.
4.  **Gather Information & Resource Mobilization:** Concurrently gather more specifics (exact location of uncle, diabetic patient's last dose/condition, factory stability, precise location of 40 people) while mobilizing all necessary resources.
5.  **Logistics for Sustained Support:** Plan for food, water, and shelter once immediate threats are managed.

---

**Answer:**

**Emergency Response Plan: Kandy Landslide - Hanthana Tea Factory**

1.  **Immediate Life Rescue (Uncle):**
    *   **Dispatch Swiftwater/Aerial Rescue Team:** Immediately deploy the nearest swiftwater rescue team, supported by aerial assets (helicopter) if available and safe, to the uncle's precise location. Emphasize speed due to rising water.
    *   **Direct Factory Contacts:** Instruct those inside the factory to maintain visual contact with the uncle if possible and relay any changes in his condition or water level to responders.

2.  **Immediate Medical Intervention (Diabetic Patient):**
    *   **Dispatch Medical Team with Insulin:** Mobilize a medical team equipped with insulin and other emergency medical supplies.
    *   **Assess Access for Medical Delivery:** Determine the quickest route:
        *   **Aerial Drop:** If the factory is inaccessible by ground, attempt a precise aerial drop of medical personnel and insulin/supplies.
        *   **Foot Team:** If partial foot access is possible, deploy a rapid response medical foot team.
    *   **Telemedicine Support:** Connect factory personnel with medical professionals via satellite phone/radio for remote first aid instructions until help arrives.

3.  **Access Road Assessment & Clearing:**
    *   **Rapid Assessment Team:** Deploy an engineering/ground assessment team to determine the exact nature and extent of the road blockage.
    *   **Heavy Equipment Mobilization:** Simultaneously request heavy machinery (excavators, bulldozers) to commence clearing operations from the most accessible point. Prioritize creating a passable lane for emergency vehicles.

4.  **Information Gathering & Resource Coordination:**
    *   **Establish Communication:** Secure a reliable communication link (satellite phone, radio) with the factory.
    *   **Detailed Status Report:** Obtain full details on:
        *   Number of people and their condition inside the factory (injuries, other medical needs).
        *   Structural integrity of the factory building.
        *   Location and condition of the 40 hungry people.
        *   Any other immediate hazards (e.g., further landslide risk, rising water inside).
    *   **Notify Hospitals:** Alert local hospitals to prepare for potential casualties from both the landslide and the medical emergencies.

5.  **Logistics & Sustained Support (40 People Hungry):**
    *   **Prepare Food & Water Drop:** Assemble emergency food rations, potable water, and blankets for an aerial drop or delivery via foot teams once safe access is established.
    *   **Shelter Assessment:** Assess the need for temporary shelter once evacuation or prolonged stay is determined, considering the factory's safety.
    *   **Psychological First Aid:** Prepare for psychological support for the trapped individuals once immediate threats are mitigated.


--- Chaos Run 2 ---


...


--- Chaos Run 2 ---


**Reasoning Steps:**

1.  **Assess Immediate Life Threat:** The 3 ICU patients on ventilators face a certain and imminent threat to life within 2 hours if power is not restored. The 50 elderly bedridden patients are in a high-risk situation due to floodwaters, but the threat, while severe, is less immediately and directly tied to the *loss of power* as the primary cause of death compared to ventilator failure.
2.  **Evaluate Resource vs. Need:** The generator directly addresses the critical power need for the ICU ventilators, preventing immediate loss of life. While power could assist the ground floor ward, its primary problem is flooding and the need for evacuation, which a generator alone does not solve.
3.  **Prioritize Intervention:** The generator must be allocated to where it can prevent the most immediate and certain loss of life due to its specific function. This is the ICU.
4.  **Concurrent Action:** A separate, immediate, and high-priority action must be initiated for the ground floor ward, focusing on evacuation.

---

**Answer:**

1.  **Direct the Generator:** Immediately upon arrival (in 30 minutes), the generator truck must be routed directly and exclusively to the **Intensive Care Unit (ICU)** to restore power for the 3 patients on ventilators, as their battery backup is critical and time-limited.
2.  **Initiate Ground Floor Evacuation:** Simultaneously, launch an immediate, full-scale evacuation of the 50 elderly, bedridden patients from the ground floor ward.
    *   Mobilize all available hospital staff (medical, nursing, support, administrative) to assist with patient transfer.
    *   Request urgent external support (e.g., fire department, emergency services, military, volunteers) for additional manpower and transport resources.
    *   Identify and prepare higher-level, safe wards or areas within the hospital for patient relocation.
    *   Prioritize patients during evacuation based on their medical stability and needs.


--- Chaos Run 3 ---


**Reasoning Steps:**

1.  **Evaluate Immediacy and Direct Threat to Life:** The 3 ICU patients on ventilators have a 2-hour battery backup, facing imminent death if power isn't restored. This is the most time-critical and direct life-threatening situation.
2.  **Assess Resource Effectiveness:** The generator can directly resolve the ICU power failure, preventing immediate loss of life. While floodwater is a severe risk to 50 elderly patients, the generator cannot stop the flood; its benefit to the flooded ward would be secondary (e.g., lighting for evacuation) compared to directly maintaining life support.
3.  **Prioritize Core Life-Saving Function:** The primary role of emergency power in this scenario is to sustain life support functions where no immediate alternative exists.
4.  **Initiate Concurrent Actions:** Addressing the flooding requires immediate human intervention for evacuation, which should be initiated simultaneously and with maximum urgency, independent of the generator allocation.

---

**Answer:**

1.  **Direct Generator to ICU:** Immediately instruct the arriving generator truck to power the Intensive Care Unit (ICU). This is the absolute priority to maintain life support for the 3 critical patients on ventilators, whose battery backup will fail within 2 hours, leading to certain death.
2.  **Initiate Immediate Evacuation of Ground Floor:** Simultaneously, trigger a full-scale emergency evacuation protocol for the 50 bedridden elderly patients from the ground floor ward. Mobilize all available hospital staff, request immediate assistance from local emergency services (Fire Department, Police, Disaster Management), and community volunteers for patient relocation to higher, safe floors within the hospital or designated safe zones.
3.  **Monitor ICU Status:** Continuously monitor the battery levels of ICU ventilators until generator power is fully established.
4.  **Assess Flood Progression:** Monitor floodwater levels and progression to anticipate further impact on hospital infrastructure and patient safety.


################################################################################

